# Documentação e tags da gold (etapa 4.4)

Aplica **comentários de coluna** e **tags** nas 12 tabelas da camada gold (`olist.gold`): 4 dimensões, 3 fatos e 5 agregados. É a base que a Fase 6 (governança e LGPD) usa para mascarar colunas e conceder acesso por perfil, e é o que o Genie e o Catalog Explorer mostram para quem consulta.

**Tags usadas** (as mesmas da silver, mais duas de tabela)

| Tag | Onde | Valores | Significado |
|---|---|---|---|
| `camada` | tabela | `gold` | camada do medalhão |
| `tipo_tabela` | tabela | `dimensao`, `fato`, `agregado` | papel no modelo estrela |
| `contem_pii` | tabela | `true` / `false` | a tabela tem alguma coluna com `pii = true` |
| `pii` | coluna | `true` | a coluna é dado pessoal ou permite identificar uma pessoa |
| `pii_categoria` | coluna | `identificador`, `localizacao` | tipo do dado pessoal; a Fase 6 escolhe a máscara por categoria |

**Decisões de classificação** (mesmas da silver, ADR 0006)

- `customer_unique_id` é o identificador estável da pessoa: PII, `identificador` (em `dim_customer` e `fact_orders`).
- Cidade, UF e prefixo de CEP do **cadastro** (`dim_customer`, `dim_seller`, `gold_seller_scorecard`) são PII `localizacao`; em vendedores é a localização da empresa, e a Fase 6 decide se mascara.
- `customer_sk` e `seller_sk` são hashes de id + início da versão: chaves técnicas, não PII.
- Agregados por UF do cliente (`gold_delivery_sla`) são contagens de muitos pedidos, não identificam ninguém: sem tag.
- O texto do comentário da review **não existe** na gold (só `tem_comentario`).

**Como aplicar nas materialized views:** o comentário de **coluna** usa `ALTER MATERIALIZED VIEW ... ALTER COLUMN ... COMMENT` (o `ALTER TABLE` recusa, porque a materialized view é uma *view*), e as tags usam `ALTER TABLE ... SET TAGS`, na tabela e na coluna. O comentário de **tabela** não é aplicado aqui: ele é declarado no pipeline (`comment=` de cada tabela) e o pipeline o regrava a cada atualização, desfazendo qualquer `COMMENT ON TABLE`. Nos testes o comentário de coluna sobreviveu a uma atualização do pipeline; as tags você confere na última célula, depois de rodar o pipeline.

**Idempotente:** comentários e tags sobrescrevem o valor anterior.

In [ ]:
# O QUE FAZ: define o catálogo e, para cada tabela da gold, a descrição de cada coluna, as tags de PII e as
#   tags de tabela. Nada é gravado aqui.
#
# RECURSOS:
# - dict {tabela: {"tabela": (comentário, tags), "colunas": {coluna: (comentário, tags)}}}: a documentação
#   fica em dados, e o código que a aplica (próxima célula) é um só para as 12 tabelas.
# - tags: dict {chave: valor}; vazio {} = a coluna não é PII.
# - função tabela(tipo, comentario, colunas): monta o item de uma tabela e acrescenta as tags camada/tipo_tabela/
#   contem_pii. O 'comentario' espelha o texto declarado no pipeline e NÃO é aplicado (o pipeline é o dono dele).
CATALOG = "olist"

PII_IDENTIFICADOR = {"pii": "true", "pii_categoria": "identificador"}
PII_LOCALIZACAO = {"pii": "true", "pii_categoria": "localizacao"}


def tabela(tipo, comentario, colunas):
    contem_pii = any(t.get("pii") == "true" for _, t in colunas.values())
    return {
        "tabela": (comentario, {"camada": "gold", "tipo_tabela": tipo, "contem_pii": "true" if contem_pii else "false"}),
        "colunas": colunas,
    }


DOCUMENTACAO = {
    "gold.dim_date": tabela("dimensao", "Calendário: uma linha por dia, da primeira compra até 30 dias depois da última entrega estimada. Chave date_key (aaaammdd).", {
        "date_key": ("Chave do dia no formato aaaammdd (inteiro). Chave da tabela.", {}),
        "data": ("Data do dia.", {}),
        "ano": ("Ano.", {}),
        "trimestre": ("Trimestre do ano (1 a 4).", {}),
        "mes": ("Mês do ano (1 a 12).", {}),
        "ano_mes": ("Mês no formato aaaa-mm.", {}),
        "dia": ("Dia do mês.", {}),
        "dia_da_semana": ("Dia da semana, de 1 (domingo) a 7 (sábado).", {}),
        "nome_dia_semana": ("Nome do dia da semana em inglês.", {}),
        "fim_de_semana": ("Verdadeiro para sábado e domingo.", {}),
    }),
    "gold.dim_product": tabela("dimensao", "Produtos com a categoria em inglês. categoria_origem diz se veio da tradução, do nome original em português ou se é desconhecida.", {
        "product_id": ("Código do produto. Chave da tabela.", {}),
        "categoria": ("Categoria em inglês; nome original em português quando não há tradução; desconhecida quando o produto não tem categoria.", {}),
        "categoria_pt": ("Nome original da categoria, em português. Nulo quando o produto não tem categoria.", {}),
        "categoria_origem": ("De onde veio a coluna categoria: traduzida, original_pt (sem tradução) ou desconhecida.", {}),
        "peso_g": ("Peso do produto em gramas.", {}),
        "comprimento_cm": ("Comprimento da embalagem em centímetros.", {}),
        "altura_cm": ("Altura da embalagem em centímetros.", {}),
        "largura_cm": ("Largura da embalagem em centímetros.", {}),
        "fotos_qtd": ("Quantidade de fotos do anúncio.", {}),
    }),
    "gold.dim_customer": tabela("dimensao", "Clientes por versão do cadastro (SCD2): uma linha por versão, com customer_sk e o intervalo de validade. Contém dados pessoais, inclusive endereços antigos.", {
        "customer_sk": ("Chave substituta da versão: hash SHA-256 de customer_id + valido_de. Chave da tabela. Não é PII.", {}),
        "customer_id": ("Código do cliente no pedido (um por pedido). Não é PII.", {}),
        "customer_unique_id": ("Identificador estável da pessoa. Dado pessoal.", PII_IDENTIFICADOR),
        "customer_city": ("Cidade do cliente nesta versão. Dado pessoal.", PII_LOCALIZACAO),
        "customer_state": ("UF do cliente nesta versão. Dado pessoal.", PII_LOCALIZACAO),
        "customer_zip_code_prefix": ("Prefixo do CEP nesta versão (texto, 5 dígitos). Dado pessoal.", PII_LOCALIZACAO),
        "valido_de": ("Início da validade da versão (inclusive).", {}),
        "valido_ate": ("Fim da validade da versão (exclusive). Nulo = versão vigente ou cadastro excluído sem versão aberta.", {}),
        "versao_vigente": ("Verdadeiro quando valido_ate é nulo.", {}),
        "cadastro_excluido_hoje": ("Verdadeiro quando o cadastro foi excluído (nenhuma versão aberta). Os pedidos continuam existindo; a Fase 6 decide sobre os dados pessoais.", {}),
    }),
    "gold.dim_seller": tabela("dimensao", "Vendedores por versão do cadastro (SCD2): uma linha por versão, com seller_sk e o intervalo de validade.", {
        "seller_sk": ("Chave substituta da versão: hash SHA-256 de seller_id + valido_de. Chave da tabela. Não é PII.", {}),
        "seller_id": ("Código do vendedor.", {}),
        "seller_city": ("Cidade do vendedor nesta versão. Localização.", PII_LOCALIZACAO),
        "seller_state": ("UF do vendedor nesta versão. Localização.", PII_LOCALIZACAO),
        "seller_zip_code_prefix": ("Prefixo do CEP do vendedor nesta versão (texto, 5 dígitos). Localização.", PII_LOCALIZACAO),
        "valido_de": ("Início da validade da versão (inclusive).", {}),
        "valido_ate": ("Fim da validade da versão (exclusive). Nulo = versão vigente ou cadastro excluído sem versão aberta.", {}),
        "versao_vigente": ("Verdadeiro quando valido_ate é nulo.", {}),
        "cadastro_excluido_hoje": ("Verdadeiro quando o cadastro foi excluído (nenhuma versão aberta). As vendas continuam existindo.", {}),
    }),
    "gold.fact_orders": tabela("fato", "Fato de pedidos: uma linha por pedido, com status atual e data de entrega derivados dos eventos, flags (pedido_valido, entregue, entregue_no_prazo), GMV, lead time, review mais recente e a versão do cliente na compra.", {
        "order_id": ("Código do pedido. Chave da tabela.", {}),
        "customer_id": ("Código do cliente no pedido. Não é PII.", {}),
        "customer_sk": ("Versão do cadastro do cliente vigente no instante da compra (junção com dim_customer).", {}),
        "customer_unique_id": ("Identificador estável da pessoa, da versão do cadastro na compra. Dado pessoal.", PII_IDENTIFICADOR),
        "cliente_excluido_hoje": ("Verdadeiro quando o cadastro do cliente foi excluído depois da compra.", {}),
        "order_purchase_timestamp": ("Instante da compra, no relógio local do Brasil (sem conversão de fuso).", {}),
        "order_purchase_date": ("Data da compra: período de referência de todas as métricas.", {}),
        "date_key": ("Chave da data da compra (aaaammdd), junção com dim_date.", {}),
        "order_estimated_delivery_date": ("Data estimada de entrega informada ao cliente.", {}),
        "order_delivered_customer_date": ("Instante da entrega: event_ts do evento delivered. Nulo se não houve.", {}),
        "status_atual": ("Status do evento de maior event_sequence; desconhecido quando o pedido não tem evento.", {}),
        "qtd_itens": ("Quantidade de itens do pedido.", {}),
        "qtd_vendedores": ("Quantidade de vendedores diferentes no pedido.", {}),
        "valor_itens": ("Soma dos preços dos itens, em reais.", {}),
        "valor_frete": ("Soma dos fretes dos itens, em reais.", {}),
        "gmv_bruto": ("valor_itens + valor_frete, de qualquer pedido (válido ou não). O GMV oficial soma só pedidos válidos.", {}),
        "tem_item": ("Verdadeiro quando o pedido tem ao menos 1 item.", {}),
        "pedido_valido": ("Tem ao menos 1 item e o status atual não é canceled nem unavailable (ADR 0002).", {}),
        "pedidos_validos_do_cliente": ("Pedidos válidos do mesmo cliente único em todo o histórico; base da taxa de recompra.", {}),
        "cancelado": ("Status atual canceled.", {}),
        "indisponivel": ("Status atual unavailable.", {}),
        "entregue": ("Status atual delivered com data de entrega preenchida.", {}),
        "entregue_no_prazo": ("Só para pedidos entregues: DATE(entrega) <= DATE(entrega estimada). Nulo nos demais.", {}),
        "lead_time_dias": ("Dias, com fração, entre a compra e a entrega. Só para pedidos entregues.", {}),
        "atraso_dias": ("DATE(entrega) - DATE(entrega estimada) em dias; positivo = atrasou. Só para pedidos entregues.", {}),
        "review_id": ("Review mais recente do pedido (maior review_answer_timestamp).", {}),
        "review_score": ("Nota de 1 a 5 da review mais recente do pedido.", {}),
    }),
    "gold.fact_order_items": tabela("fato", "Fato de itens: uma linha por item de pedido, com preço, frete, item_total, status e pedido_valido do pedido e a versão do vendedor na compra (seller_sk).", {
        "order_id": ("Código do pedido. Parte da chave.", {}),
        "order_item_id": ("Número do item dentro do pedido. Parte da chave.", {}),
        "product_id": ("Código do produto (junção com dim_product).", {}),
        "seller_id": ("Código do vendedor.", {}),
        "seller_sk": ("Versão do cadastro do vendedor vigente no instante da compra (junção com dim_seller).", {}),
        "vendedor_excluido_hoje": ("Verdadeiro quando o cadastro do vendedor foi excluído depois da venda.", {}),
        "order_purchase_timestamp": ("Instante da compra do pedido.", {}),
        "order_purchase_date": ("Data da compra: período de referência.", {}),
        "date_key": ("Chave da data da compra (aaaammdd), junção com dim_date.", {}),
        "shipping_limit_date": ("Prazo limite para o vendedor enviar o item.", {}),
        "price": ("Preço do item, em reais.", {}),
        "freight_value": ("Frete do item, em reais.", {}),
        "item_total": ("price + freight_value. O GMV soma item_total dos itens de pedidos válidos.", {}),
        "status_atual": ("Status atual do pedido.", {}),
        "pedido_valido": ("Pedido válido (ADR 0002), herdado do pedido.", {}),
    }),
    "gold.fact_reviews": tabela("fato", "Fato de reviews: uma linha por par (review, pedido). eh_mais_recente marca a review que vale por pedido; pedido_existe falso são reviews sem pedido. Sem o texto do comentário.", {
        "review_id": ("Código da review. Parte da chave (o mesmo review_id pode aparecer em pedidos diferentes).", {}),
        "order_id": ("Código do pedido avaliado. Parte da chave.", {}),
        "pedido_existe": ("Falso quando o pedido não existe na silver (review órfã, fora das métricas).", {}),
        "eh_mais_recente": ("Verdadeiro para a review que vale no pedido: maior review_answer_timestamp, desempate por review_creation_date e review_id.", {}),
        "review_score": ("Nota de 1 a 5.", {}),
        "review_creation_date": ("Data de criação da review.", {}),
        "review_answer_timestamp": ("Instante da resposta da review.", {}),
        "tem_comentario": ("Verdadeiro quando a review tem texto de comentário (o texto não vai para a gold).", {}),
    }),
    "gold.gold_seller_mensal": tabela("agregado", "GMV por vendedor e mês da compra (só pedidos válidos), com rank_no_mes. Base da métrica de concentração de vendas.", {
        "ano_mes": ("Mês da compra, no formato aaaa-mm.", {}),
        "seller_id": ("Código do vendedor.", {}),
        "gmv": ("GMV do vendedor no mês (preço + frete de itens de pedidos válidos).", {}),
        "pedidos": ("Pedidos distintos do vendedor no mês.", {}),
        "rank_no_mes": ("Posição do vendedor no mês pelo GMV (1 = maior); empate decidido por seller_id.", {}),
    }),
    "gold.gold_seller_scorecard": tabela("agregado", "Scorecard de vendedores: uma linha por versão do cadastro (seller_sk). Um pedido entra em cada vendedor que tem item nele; o GMV soma só os itens do próprio vendedor. A cidade e a UF são as da data da venda.", {
        "seller_sk": ("Versão do cadastro do vendedor. Chave da tabela.", {}),
        "seller_id": ("Código do vendedor.", {}),
        "seller_city": ("Cidade do vendedor na data da venda. Localização.", PII_LOCALIZACAO),
        "seller_state": ("UF do vendedor na data da venda. Localização.", PII_LOCALIZACAO),
        "seller_zip_code_prefix": ("Prefixo do CEP do vendedor na data da venda. Localização.", PII_LOCALIZACAO),
        "valido_de": ("Início da validade da versão.", {}),
        "valido_ate": ("Fim da validade da versão (exclusive). Nulo = vigente.", {}),
        "versao_vigente": ("Verdadeiro quando valido_ate é nulo.", {}),
        "cadastro_excluido_hoje": ("Verdadeiro quando o cadastro foi excluído.", {}),
        "pedidos": ("Pedidos com ao menos um item desta versão do vendedor, em qualquer status. A soma entre vendedores passa do total de pedidos (pedidos compartilhados).", {}),
        "pedidos_validos": ("Dos pedidos acima, os que são válidos.", {}),
        "pedidos_compartilhados": ("Dos pedidos acima, os que têm itens de mais de um vendedor.", {}),
        "itens": ("Itens vendidos nesta versão, em qualquer status.", {}),
        "itens_validos": ("Itens de pedidos válidos.", {}),
        "gmv": ("GMV da versão: preço + frete dos itens de pedidos válidos.", {}),
        "pedidos_cancelados": ("Pedidos com status atual canceled.", {}),
        "taxa_cancelamento": ("pedidos_cancelados / pedidos.", {}),
        "pedidos_entregues": ("Pedidos entregues.", {}),
        "entregues_no_prazo": ("Pedidos entregues com DATE(entrega) <= DATE(estimada).", {}),
        "pct_entregas_no_prazo": ("entregues_no_prazo / pedidos_entregues. Nulo sem entregas.", {}),
        "atraso_medio_dias": ("Média de atraso_dias dos pedidos entregues; positivo = atrasou.", {}),
        "lead_time_medio_dias": ("Média de dias entre compra e entrega dos pedidos entregues.", {}),
        "pedidos_com_review": ("Pedidos entregues com review.", {}),
        "review_score_medio": ("Média da nota da review mais recente dos pedidos entregues.", {}),
    }),
    "gold.gold_delivery_sla": tabela("agregado", "SLA de entrega por mês da compra e UF do cliente na compra, só pedidos entregues. Contagens guardadas para reagregar.", {
        "ano_mes": ("Mês da compra, no formato aaaa-mm.", {}),
        "uf_cliente": ("UF do cliente na data da compra. Agregado de muitos pedidos, sem tag de PII.", {}),
        "pedidos_entregues": ("Pedidos entregues.", {}),
        "entregues_no_prazo": ("Entregues com DATE(entrega) <= DATE(estimada).", {}),
        "entregues_atrasados": ("Entregues com atraso_dias > 0.", {}),
        "lead_time_medio_dias": ("Média de dias entre compra e entrega.", {}),
        "lead_time_mediana_dias": ("Mediana de dias entre compra e entrega.", {}),
        "atraso_medio_dos_atrasados_dias": ("Média de atraso só entre os atrasados.", {}),
        "atraso_maximo_dias": ("Maior atraso_dias do grupo; negativo se todos chegaram adiantados.", {}),
        "pct_no_prazo": ("entregues_no_prazo / pedidos_entregues.", {}),
    }),
    "gold.gold_daily_kpis": tabela("agregado", "KPIs por dia da compra em colunas aditivas (contagens e somas). Razões se calculam somando e dividindo depois.", {
        "order_purchase_date": ("Dia da compra. Chave da tabela.", {}),
        "pedidos_criados": ("Todos os pedidos do dia.", {}),
        "pedidos_validos": ("Pedidos válidos do dia.", {}),
        "gmv": ("GMV do dia: gmv_bruto dos pedidos válidos.", {}),
        "pedidos_cancelados": ("Pedidos com status atual canceled.", {}),
        "pedidos_entregues": ("Pedidos entregues (dos comprados no dia).", {}),
        "entregues_no_prazo": ("Pedidos entregues no prazo.", {}),
        "soma_lead_time_dias": ("Soma dos dias de entrega dos pedidos entregues; dividir por pedidos_entregues dá o lead time médio.", {}),
        "reviews_de_entregues": ("Pedidos entregues com review.", {}),
        "soma_review_score": ("Soma das notas dos pedidos entregues com review; dividir por reviews_de_entregues dá a nota média.", {}),
    }),
    "gold.gold_atraso_vs_review": tabela("agregado", "Nota média da review por faixa de atraso na entrega, só pedidos entregues com review. Mostra se a nota cai quando o pedido atrasa.", {
        "faixa_atraso": ("Faixa de atraso (data da entrega - data estimada), com um número na frente para ordenar: 1_adiantado, 2_no_dia, 3_atraso_1_a_3_dias, 4_atraso_4_a_7_dias, 5_atraso_8_a_14_dias, 6_atraso_15_dias_ou_mais.", {}),
        "pedidos": ("Pedidos entregues com review na faixa.", {}),
        "review_score_medio": ("Nota média da review.", {}),
        "pedidos_nota_1_ou_2": ("Pedidos com nota 1 ou 2.", {}),
        "pedidos_nota_5": ("Pedidos com nota 5.", {}),
        "atraso_medio_dias": ("Média de atraso_dias na faixa.", {}),
        "pct_nota_1_ou_2": ("pedidos_nota_1_ou_2 / pedidos.", {}),
        "pct_nota_5": ("pedidos_nota_5 / pedidos.", {}),
    }),
}

## Aplicar comentários e tags

Para cada tabela e cada coluna:

- `ALTER TABLE t SET TAGS (...)`: tags da tabela (o comentário da tabela é o do pipeline);
- `ALTER MATERIALIZED VIEW t ALTER COLUMN c COMMENT '...'`: descrição da coluna (em materialized view o `ALTER TABLE` recusa);
- `ALTER TABLE t ALTER COLUMN c SET TAGS (...)`: tags da coluna. Coluna sem PII não recebe tag; se tivesse uma de uma execução anterior, ela seria removida com `UNSET TAGS`, assim o resultado final sempre espelha o dicionário.

Se alguma coluna do dicionário não existir na tabela, o comando falha de propósito: é melhor parar do que documentar uma coluna que não existe.

In [ ]:
# O QUE FAZ: percorre o dicionário DOCUMENTACAO e aplica as tags da tabela, o comentário de cada coluna e
#   as tags de PII de cada coluna. O comentário da tabela fica com o pipeline.
#
# FUNÇÕES DECLARADAS:
# - texto(valor): troca aspas simples por duas, para o texto caber dentro de um literal SQL.
# - tags_sql(tags): monta o trecho ('chave' = 'valor', ...) de um SET TAGS.
#
# FUNÇÕES NATIVAS:
# - spark.sql(comando): executa um comando SQL.
# - ALTER TABLE ... SET TAGS / UNSET TAGS: grava ou remove tags (Unity Catalog), na tabela ou na coluna.
# - ALTER MATERIALIZED VIEW ... ALTER COLUMN c COMMENT '...': descrição da coluna de uma materialized view.
# - dict.items(): percorre pares (chave, valor). ", ".join(lista): junta os textos com vírgula.
def texto(valor: str) -> str:
    return valor.replace("'", "''")


def tags_sql(tags: dict) -> str:
    return ", ".join(f"'{k}' = '{texto(v)}'" for k, v in tags.items())


TAGS_POSSIVEIS = ("pii", "pii_categoria")

for nome, doc in DOCUMENTACAO.items():
    t = f"{CATALOG}.{nome}"
    _, tags = doc["tabela"]
    spark.sql(f"ALTER TABLE {t} SET TAGS ({tags_sql(tags)})")

    for coluna, (comentario, tags) in doc["colunas"].items():
        spark.sql(f"ALTER MATERIALIZED VIEW {t} ALTER COLUMN {coluna} COMMENT '{texto(comentario)}'")
        if tags:
            spark.sql(f"ALTER TABLE {t} ALTER COLUMN {coluna} SET TAGS ({tags_sql(tags)})")
        else:
            lista = ", ".join(f"'{x}'" for x in TAGS_POSSIVEIS)
            spark.sql(f"ALTER TABLE {t} ALTER COLUMN {coluna} UNSET TAGS ({lista})")
    print(f"{t}: {len(doc['colunas'])} colunas documentadas")

## Conferir

Lê o que ficou gravado, direto do catálogo, e confere se **toda coluna das 12 tabelas** tem comentário (a lista de colunas sem comentário tem que vir vazia) e se as tags de PII estão onde deveriam.

**Esta célula roda sozinha** (ela não depende das células anteriores, então funciona numa sessão nova): rode-a agora e depois de cada execução do pipeline `olist.gold`. Ela lista as 12 materialized views direto do catálogo e mostra o que ficou gravado. Se depois de um *Full refresh* os comentários ou as tags tiverem sumido, rode o notebook inteiro de novo: o pipeline recria as tabelas e a documentação é reaplicada por este notebook.

In [ ]:
# O QUE FAZ: mostra as tags de PII gravadas, as tags de tabela e lista colunas sem comentário (deve vir vazio).
#   É autossuficiente: define o catálogo e descobre as tabelas no próprio catálogo, para rodar numa sessão nova.
#
# FUNÇÕES NATIVAS:
# - spark.table("olist.information_schema.column_tags"): tags de coluna (schema_name, table_name,
#   column_name, tag_name, tag_value). "table_tags": o mesmo para tabelas.
# - spark.sql("DESCRIBE TABLE t"): devolve col_name, data_type e comment de cada coluna.
# - coluna.isin(lista): verdadeiro se o valor está na lista; a lista vem do catálogo (tabelas do tipo MATERIALIZED_VIEW).
# - spark.table("olist.information_schema.tables"): as tabelas do catálogo, com table_schema e table_type.
# - DataFrame.where / orderBy / select: filtrar, ordenar, escolher colunas.
# - display(df): mostra o DataFrame como tabela interativa.
from pyspark.sql import functions as F

CATALOG = "olist"
# as 12 tabelas da gold são materialized views; as Metric Views (mv_*) têm outro tipo e as tabelas internas
# do pipeline (__materialization_*, event_log_*) não entram
nomes = [
    r.table_name
    for r in spark.table(f"{CATALOG}.information_schema.tables")
    .where((F.col("table_schema") == "gold") & (F.col("table_type") == "MATERIALIZED_VIEW"))
    .select("table_name")
    .collect()
]
print(f"{len(nomes)} materialized views na gold: {sorted(nomes)}")

display(
    spark.table(f"{CATALOG}.information_schema.table_tags")
    .where((F.col("schema_name") == "gold") & F.col("table_name").isin(nomes))
    .select("table_name", "tag_name", "tag_value")
    .orderBy("table_name", "tag_name")
)

display(
    spark.table(f"{CATALOG}.information_schema.column_tags")
    .where((F.col("schema_name") == "gold") & F.col("table_name").isin(nomes))
    .select("table_name", "column_name", "tag_name", "tag_value")
    .orderBy("table_name", "column_name", "tag_name")
)

for nome in sorted(nomes):
    sem_comentario = (
        spark.sql(f"DESCRIBE TABLE {CATALOG}.gold.{nome}")
        .where(F.col("col_name").isNotNull() & ~F.col("col_name").startswith("#") & (F.col("col_name") != ""))
        .where(F.col("comment").isNull() | (F.col("comment") == ""))
        .select("col_name")
    )
    print(f"gold.{nome}: colunas sem comentário -> {[r.col_name for r in sem_comentario.collect()]}")